In [ ]:
# @title Join the uploaded cache parts and check them
import glob, hashlib, os
out = "/content/f3dr/baselines/out/atlas"
want = {'train.npz': (1007472814, 'c5b97585935e9a2a8320820f6b7bfaca'), 'train_dinov3l.npz': (1075129838, '6f2ec68acca504073fd4b49035518b98')}
for name, (size, md5) in want.items():
    dst = f"{out}/{name}"
    parts = sorted(glob.glob(f"/content/parts/{name}.part*"))
    if parts and not (os.path.exists(dst) and os.path.getsize(dst) == size):
        with open(dst, "wb") as f:
            for p in parts:
                f.write(open(p, "rb").read())
    h = hashlib.md5(open(dst, "rb").read()).hexdigest()
    print(name, os.path.getsize(dst), "OK" if (os.path.getsize(dst), h) == (size, md5) else "MISMATCH")


In [ ]:
# @title Packages and GPU
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "trimesh"], check=True)
import torch; print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
# @title Smoke test: 1 epoch with the new position embedding
import subprocess
r = subprocess.run("cd /content/f3dr && python baselines/atlas.py train --target mm --no-flip-loss --ema 0.999 --encoder dinov3l --arch talk --pos-emb --epochs 1 --ckpt /tmp/smoke.pt",
                   shell=True, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:], "rc", r.returncode)
